In [1]:
from firedrake import *

In [ ]:
omega = IntervalMesh(10,1)


"""The geometric map from reference cell: assigns a point in physical space to each reference point."""
print(omega.coordinates.function_space()) # mesh coordinate field is a vector field


WithGeometry(FunctionSpace(<firedrake.mesh.MeshTopology object at 0x12f253530>, VectorElement(FiniteElement('Lagrange', interval, 1), dim=1), name=None), <Mesh #68>)


In [38]:
# Function spaces
P1CG = FunctionSpace(omega, "CG", 1)
P2CG = FunctionSpace(omega, "CG", 2)

In [ ]:
u = TrialFunction(P1CG)
v = TestFunction(P1CG)

bilinear_form = u * v * dx
assembled_bilinear_form = assemble(bilinear_form) # returns the mass matrix associated with the CG1 finite element basis

print(type(assembled_bilinear_form)) # returns a `Matrix` object
print(assembled_bilinear_form)

assembled_petsc_mat = assembled_bilinear_form.petscmat
assembled_petsc_mat.view() # in this case this corresponds to the mass matrix of the L2 inner product

<class 'firedrake.matrix.Matrix'>
assembled Matrix(a=v_0 * v_1 * dx(<Mesh #2>[everywhere], {}, {}), bcs=())
Mat Object: 1 MPI process
  type: seqaij
row 0: (0, 0.0333333)  (1, 0.0166667) 
row 1: (0, 0.0166667)  (1, 0.0666667)  (2, 0.0166667) 
row 2: (1, 0.0166667)  (2, 0.0666667)  (3, 0.0166667) 
row 3: (2, 0.0166667)  (3, 0.0666667)  (4, 0.0166667) 
row 4: (3, 0.0166667)  (4, 0.0666667)  (5, 0.0166667) 
row 5: (4, 0.0166667)  (5, 0.0666667)  (6, 0.0166667) 
row 6: (5, 0.0166667)  (6, 0.0666667)  (7, 0.0166667) 
row 7: (6, 0.0166667)  (7, 0.0666667)  (8, 0.0166667) 
row 8: (7, 0.0166667)  (8, 0.0666667)  (9, 0.0166667) 
row 9: (8, 0.0166667)  (9, 0.0666667)  (10, 0.0166667) 
row 10: (9, 0.0166667)  (10, 0.0333333) 


In [ ]:
u = TestFunction(P1CG)
v = Function(P1CG)

one_linear_form = u * v * dx
assembled_one_linear_form = assemble(one_linear_form)

print(type(assembled_one_linear_form)) # returns a Cofunction since 1-linear forms are mathematically equivalent to covectors
print(assembled_one_linear_form.dat.data[:])


<class 'firedrake.cofunction.Cofunction'>
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [23]:
u = Function(P1CG)
v = Function(P1CG)

zero_linear_form = u * v * dx
assembled_zero_linear_form = assemble(zero_linear_form)

print(type(assembled_zero_linear_form))
print(assembled_zero_linear_form)

<class 'numpy.float64'>
0.0


In [ ]:
"""
Evaluate the L^2 inner product of two functions in CG2
"""

u = Function(P2CG)
v = Function(P1CG)

x, = SpatialCoordinate(omega) # since the mesh coordinate field is a vector field, this returns a vector of symbolic coordinate variables
u.interpolate(x ** 2)
v.interpolate(2*x)

u_dot_v = assemble(u * v * dx)
print(u_dot_v)

0.4999999999999999


Observation: When interpolating into CG1, the evaluation nodes coincide with the mesh points. Given that `SpatialCoordinate` returns a vector of symbolic coordinate variables, interpolating `2 * x` into CG1 raises a `shape mismatch` error: `2 * x` is a vector field while CG1 is a scalar field.
When interpolating into CG2 instead, we do not get the same error. The difference here is that the evaluation nodes are no longer the exact mesh points. This causes the program to execute a different code path which automatically converts the vector field into the required scalar type. 